# LRURec：算法、评估与推荐

先运行 `python train.py --model lrurec`。本 Notebook 展示验证选出的模型，并与 ItemCF、LightFM 使用相同规则比较。

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / 'src' / 'models' / 'lrurec.py').is_file())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.recommend import file_sha256, recommend_for_user
from src.evaluate import write_comparison

TABLE_DIR = PROJECT_ROOT / 'outputs' / 'tables'
run_path = TABLE_DIR / 'lrurec_run.json'
if not run_path.is_file():
    raise FileNotFoundError('请先运行 python train.py --model lrurec。')
run = json.loads(run_path.read_text(encoding='utf-8'))
for name, checksum in run['input_sha256'].items():
    if file_sha256(PROJECT_ROOT / 'data' / 'processed' / name) != checksum:
        raise ValueError('预处理输入已变化，请重新训练 LRURec。')

## 1. 算法与选定参数

LRURec 将高分电影序列编码为向量，用线性递推累积历史，再预测下一部高分电影。核心包含复数对角递推、残差前馈层与共享电影嵌入。
训练标签仅来自训练集，相邻同时间戳的预测标签被忽略；推理仅取最近的已知高分历史。

下面展示已完成实验记录的选模规则、参数和代表性轮数。新的训练按最低验证交叉熵 loss 选模；验证标签不参与参数更新。
验证 loss 只使用训练期和此前验证期的高分历史。旧实验若未记录验证 loss，则只展示已有指标。

In [2]:
print("Selection:", run['selection_metric'])
display(pd.DataFrame([run['selected_parameters']]))
search = pd.read_csv(TABLE_DIR / 'lrurec_validation_search.csv')
checkpoints = {1, 3, 5, 10, 20, run['selected_parameters']['epochs']}
columns = [name for name in ['epochs', 'train_loss', 'validation_loss', 'NDCG@10', 'selected']
           if name in search.columns]
display(search.loc[search.epochs.isin(checkpoints), columns]
        .style.format({'train_loss': '{:.4f}', 'validation_loss': '{:.4f}', 'NDCG@10': '{:.4f}'}))

Selection: NDCG@10 on validation


,n_items,hidden_size,num_blocks,max_length,dropout,epochs
0,3667,32,2,50,0.2,3


,epochs,train_loss,NDCG@10,selected
0,1,7.3589,0.0533,False
1,3,6.4541,0.1098,True


## 2. 三种算法比较

使用共同评估用户与完整未评分训练目录，先按用户计算再取平均。
Precision@10＝命中数 /10；Recall@10＝命中数 /高分目标数；NDCG@10 考虑命中位置；MeanHits@10 是平均命中部数。
未评分电影的偏好未知，不计为离线已知命中。结果代表当前参数与轮数搜索范围，不能说明算法的最佳表现。

In [3]:
write_comparison(TABLE_DIR)
comparison = pd.read_csv(TABLE_DIR / 'comparison_metrics.csv')
display(comparison[['algorithm', 'split', 'users', 'Precision@10', 'Recall@10', 'NDCG@10', 'MeanHits@10']]
        .style.format({'Precision@10': '{:.2%}', 'Recall@10': '{:.2%}',
                       'NDCG@10': '{:.4f}', 'MeanHits@10': '{:.3f}'}))

,algorithm,split,users,Precision@10,Recall@10,NDCG@10,MeanHits@10
0,ItemCF,validation,5776,5.97%,7.89%,0.0844,0.597
1,ItemCF,test,5765,5.64%,7.12%,0.0773,0.564
2,LightFM,validation,5776,4.22%,4.36%,0.0523,0.422
3,LightFM,test,5765,4.11%,4.18%,0.0518,0.411
4,LRURec,validation,5776,7.20%,11.42%,0.1098,0.720
5,LRURec,test,5765,6.46%,10.01%,0.0961,0.646


## 3. 按用户比较推荐电影

修改 `USER_ID`，每种算法推荐 5 部历史未评分电影，展示标题与类型。

In [4]:
USER_ID = 1
TOP_K = 5

recommendations = []
for algorithm in ['itemcf', 'lightfm', 'lrurec']:
    result = recommend_for_user(USER_ID, stage='test', k=TOP_K, algorithm=algorithm)
    result.insert(0, 'algorithm', algorithm)
    recommendations.append(result[['algorithm', 'rank', 'title', 'genres']])
display(pd.concat(recommendations, ignore_index=True))

,algorithm,rank,title,genres
0,itemcf,1,Raiders of the Lost Ark (1981),Action|Adventure
1,itemcf,2,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
2,itemcf,3,"Shawshank Redemption, The (1994)",Drama
3,itemcf,4,"Silence of the Lambs, The (1991)",Drama|Thriller
4,itemcf,5,Pulp Fiction (1994),Crime|Drama
5,lightfm,1,Raiders of the Lost Ark (1981),Action|Adventure
6,lightfm,2,"Shawshank Redemption, The (1994)",Drama
7,lightfm,3,"Godfather, The (1972)",Action|Crime|Drama
8,lightfm,4,Star Wars: Episode V - The Empire Strikes Back...,Action|Adventure|Drama|Sci-Fi|War
9,lightfm,5,"Fugitive, The (1993)",Action|Thriller


参考：[LRURec: Sequential Recommendation with Linear Recurrent Units (WSDM 2024)](https://arxiv.org/abs/2310.02367)；[官方代码](https://github.com/yueqirex/LRURec)。本项目独立实现核心结构，并适配统一时间划分与评估。